In [21]:
import re
import numpy as np

In [22]:
text = open('pg100.txt','r',encoding='utf-8',errors='ignore').read()
text = re.sub(r'[^\x20-\x7E\n]','',text)
text = re.sub(r'[ \t]+',' ',text)
text = re.sub(r'\n{3,}','\n\n',text)
text = text.strip()

In [23]:
chars = sorted(set(text))
vocab_size = len(chars)

In [24]:
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:s for i,s in enumerate(chars)}

In [25]:
encoder = lambda s: [stoi[c] for c in s]
decoder = lambda l: ''.join([itos[i] for i in l])

In [26]:
encoder("Hi")

[36, 66]

In [27]:
data = np.array(encoder(text), dtype=np.int32)
split = int(0.9*len(data))
train = data[:split]
test = data[split:]

In [28]:
def get_batch(data,block_size = 64,batch_size = 32):
    ix = np.random.randint(0,len(data)-block_size,size=batch_size)
    x = np.stack([data[i:i+block_size] for i in ix])
    y = np.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

In [29]:
from collections import Counter

def get_pairs(vocab):
    pairs = {}
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pair = (symbols[i], symbols[i+1])
            pairs[pair] = pairs.get(pair, 0) + freq
    return pairs

def merge_pair(pair, vocab):
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab

word_freq = Counter(text.split())
vocab = {' '.join(list(w)) + ' </w>': f for w, f in word_freq.items()}

merges = []
for i in range(500):
    pairs = get_pairs(vocab)
    best = max(pairs, key=pairs.get)
    vocab = merge_pair(best, vocab)
    merges.append(best)
    print(f"{i+1}: merged {best}")

1: merged ('e', '</w>')
2: merged ('t', 'h')
3: merged (',', '</w>')
4: merged ('.', '</w>')
5: merged ('t', '</w>')
6: merged ('s', '</w>')
7: merged ('d', '</w>')
8: merged ('e', 'r')
9: merged ('o', 'u')
10: merged ('i', 'n')
11: merged ('a', 'n')
12: merged ('y', '</w>')
13: merged ('o', 'r')
14: merged ('o', '</w>')
15: merged ('e', 'n')
16: merged ('a', 'r')
17: merged ('o', 'n')
18: merged ('l', 'l')
19: merged ('h', 'a')
20: merged ('th', 'e</w>')
21: merged ('f', '</w>')
22: merged ('i', 's</w>')
23: merged ('e', 's')
24: merged ('an', 'd</w>')
25: merged ('I', '</w>')
26: merged ('ll', '</w>')
27: merged ('y', 'ou')
28: merged ('er', '</w>')
29: merged ('e', 'a')
30: merged ('t', 'o</w>')
31: merged ('e', ',</w>')
32: merged ('o', 'w')
33: merged ('o', 'f</w>')
34: merged ('in', 'g')
35: merged ('w', 'i')
36: merged ('r', '</w>')
37: merged ('o', 'm')
38: merged ('th', '</w>')
39: merged ('s', 't')
40: merged ('a', '</w>')
41: merged ('in', '</w>')
42: merged ('h', 'i')
43: m

In [37]:
def encode_word(word,merges):
    symbols = list(word)+['</w>']
    for pair in merges:
        i = 0
        while i < len(symbols)-1:
            if (symbols[i],symbols[i+1])==pair:
                symbols = symbols[:i]+[symbols[i]+symbols[i+1]]+symbols[i+2:]
            else:
                i+=1
    return symbols

def encode(text,merges):
    return [stoi[t] for word in text.split() for t in encode_word(word,merges)]

def decode(indices):
    return ' '.join([itos[i].replace('</w>','') for i in indices])

In [44]:
vocab = set()

for word in text.split():
    tokens = encode_word(word, merges)
    vocab.update(tokens)

vocab = sorted(vocab)

stoi = {token: i for i, token in enumerate(vocab)}
itos = {i: token for i, token in enumerate(vocab)}

In [45]:
data = np.array(encode(text,merges),dtype=np.int32)
split = int(0.9*len(data))
train = data[:split]
test = data[split:]

In [46]:
x,y = get_batch(train)
print(f"train: {len(train):,} | test: {len(test):,}")
print(f"x: {x.shape} | y: {y.shape}")

vocab: 11058
train: 2,209,045 | test: 245,450
x: (32, 64) | y: (32, 64)


In [47]:
print(f"sample: {decode(train[:20].tolist())}")


sample: 1  F r om  fa ir es t c r ea t u re s we d es


In [49]:
len(stoi)

548

In [50]:
from micrograd import Value,MLP

vocab_size = 548
d_model = 64

Emb = [[Value(np.random.randn()*0.01) for _ in range(d_model)] for _ in range(vocab_size)]

def embed(token_ids):
    return Emb[token_ids]

In [51]:
def softmax(x):
    e = np.exp(x-np.max(x,axis=-1,keepdims=True))
    return e / np.sum(e, axis=-1, keepdims=True)

def attention(Q,K,V):
    d_k = Q.shape[-1]
    scores = Q @ K.T/d_k**0.5
    weights = softmax(scores)
    return weights @ V

In [55]:
def init_linear(d_in, d_out):
    w = [[Value(np.random.randn()*0.01) for _ in range(d_in)] for _ in range(d_out)]
    b = [Value(0.0) for _ in range(d_out)]
    return w, b

def linear(x, w, b):
    d_out = len(w)
    d_in = len(w[0])
    return [sum(w[i][j]*x[j] for j in range(d_in)) + b[i] for i in range(d_out)]

In [68]:
h = 4
d_k = d_model // h
seq_len = 64
W_Qs = [init_linear(d_model, d_k) for _ in range(h)]
W_Ks = [init_linear(d_model, d_k) for _ in range(h)]
W_Vs = [init_linear(d_model, d_k) for _ in range(h)]
W_O, b_O = init_linear(d_model, d_model)

def multihead(x, W_Qs, W_Ks, W_Vs, W_O, b_O):
    heads = []
    for i in range(h):
        Q = [linear(x[t], *W_Qs[i]) for t in range(seq_len)]
        K = [linear(x[t], *W_Ks[i]) for t in range(seq_len)]
        V = [linear(x[t], *W_Vs[i]) for t in range(seq_len)]
        Q_np = np.array([[v.data for v in row] for row in Q])
        K_np = np.array([[v.data for v in row] for row in K])
        V_np = np.array([[v.data for v in row] for row in V])
        head = attention(Q_np, K_np, V_np)
        heads.append(head)
    concat = [np.concatenate([heads[i][t] for i in range(h)]) for t in range(seq_len)]
    out = [linear(list(concat[t]), W_O, b_O) for t in range(seq_len)]
    return out

In [63]:
W_f1s = [[Value(np.random.randn()*0.01) for _ in range(d_model)] for _ in range(2*d_model)]
W_f2s = [[Value(0.0) for _ in range(2*d_model)] for _ in range(d_model)]
b_f1s = [Value(0.0) for _ in range(2*d_model)]
b_f2s = [Value(0.0) for _ in range(d_model)]

def relu(x):
    return [v if v.data > 0 else Value(0.0) for v in x]

def feed_forward(x,w1,b1,w2,b2):
    out_1 = linear(x,w1,b1)
    out_1 = relu(out_1)
    out_2 = linear(out_1,w2,b2)
    return out_2

In [75]:
def make_mask(seq_len):
    mask = np.zeros((seq_len, seq_len))
    mask[np.triu_indices(seq_len, k=1)] = float('-inf')
    return mask

def masked_attention(Q, K, V, mask):
    d_k = len(Q[0])
    seq_len = len(Q)
    scores = [[sum(Q[i][k] * K[j][k] for k in range(d_k))
               for j in range(seq_len)]
              for i in range(seq_len)]
    scores = [[scores[i][j] + mask[i][j]
               for j in range(seq_len)]
              for i in range(seq_len)]
    weights = [softmax(scores[i]) for i in range(seq_len)]
    out = [[sum(weights[i][j] * V[j][k] for j in range(seq_len))
            for k in range(d_k)]
           for i in range(seq_len)]
    return out

In [69]:
W_Q_cross = [init_linear(d_model, d_k) for _ in range(h)]
W_K_cross = [init_linear(d_model, d_k) for _ in range(h)]
W_V_cross = [init_linear(d_model, d_k) for _ in range(h)]

def cross_attention(decoder_x, encoder_out, W_Qs, W_Ks, W_Vs, W_O):
    heads = []
    for i in range(h):
        Q = [linear(decoder_x[t], *W_Qs[i]) for t in range(seq_len)]
        K = [linear(encoder_out[t], *W_Ks[i]) for t in range(seq_len)]
        V = [linear(encoder_out[t], *W_Vs[i]) for t in range(seq_len)]
        Q_np = np.array([[v.data for v in row] for row in Q])
        K_np = np.array([[v.data for v in row] for row in K])
        V_np = np.array([[v.data for v in row] for row in V])
        head = attention(Q_np, K_np, V_np)
        heads.append(head)
    concat = [np.concatenate([heads[i][t] for i in range(h)]) for t in range(seq_len)]
    out = [linear(list(concat[t]), W_O, b_O) for t in range(seq_len)]
    return out

In [60]:
def layer_norm(x, gamma, beta, eps=1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    x_norm = (x - mean) / np.sqrt(var + eps)
    return gamma * x_norm + beta

gamma = np.ones(d_model)
beta = np.zeros(d_model)

In [61]:
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len).reshape(-1, 1)
    i   = np.arange(0, d_model, 2).reshape(1, -1)
    denom = 10000 ** (i / d_model)
    PE = np.zeros((seq_len, d_model))
    PE[:, 0::2] = np.sin(pos / denom)
    PE[:, 1::2] = np.cos(pos / denom)
    return PE

In [62]:
E = np.random.randn(vocab_size, d_model) * 0.01

def embed(x):
    token_emb = E[x]
    pos_emb   = positional_encoding(len(x), d_model)
    return token_emb + pos_emb

In [70]:
gamma1 = np.ones(d_model)
beta1  = np.zeros(d_model)
gamma2 = np.ones(d_model)
beta2  = np.zeros(d_model)

def encoder_block(x):
    x_list = [list(x[t]) for t in range(seq_len)]
    mha_out = multihead(x_list, W_Qs, W_Ks, W_Vs, W_O, b_O)
    mha_np  = np.array([[v.data for v in row] for row in mha_out])
    x = layer_norm(x + mha_np, gamma1, beta1)
    x_list = [list(x[t]) for t in range(seq_len)]
    ff_out  = [feed_forward(x_list[t], W_f1s, b_f1s, W_f2s, b_f2s) for t in range(seq_len)]
    ff_np   = np.array([[v.data for v in row] for row in ff_out])
    x = layer_norm(x + ff_np, gamma2, beta2)
    return x

tokens = np.random.randint(0, vocab_size, size=seq_len)
x = embed(tokens)
out = encoder_block(x)
print(out.shape)

(64, 64)


In [71]:
N = 6

encoder_params = [{
    'W_Qs': [init_linear(d_model, d_k) for _ in range(h)],
    'W_Ks': [init_linear(d_model, d_k) for _ in range(h)],
    'W_Vs': [init_linear(d_model, d_k) for _ in range(h)],
    'W_O':  init_linear(d_model, d_model),
    'W_f1': [[Value(np.random.randn()*0.01) for _ in range(d_model)] for _ in range(2*d_model)],
    'W_f2': [[Value(0.0) for _ in range(2*d_model)] for _ in range(d_model)],
    'b_f1': [Value(0.0) for _ in range(2*d_model)],
    'b_f2': [Value(0.0) for _ in range(d_model)],
    'gamma1': np.ones(d_model), 'beta1': np.zeros(d_model),
    'gamma2': np.ones(d_model), 'beta2': np.zeros(d_model),
} for _ in range(N)]

def encoder_block(x, p):
    x_list = [list(x[t]) for t in range(seq_len)]
    mha_out = multihead(x_list, p['W_Qs'], p['W_Ks'], p['W_Vs'], p['W_O'][0], p['W_O'][1])
    mha_np  = np.array([[v.data for v in row] for row in mha_out])
    x = layer_norm(x + mha_np, p['gamma1'], p['beta1'])

    x_list = [list(x[t]) for t in range(seq_len)]
    ff_out  = [feed_forward(x_list[t], p['W_f1'], p['b_f1'], p['W_f2'], p['b_f2']) for t in range(seq_len)]
    ff_np   = np.array([[v.data for v in row] for row in ff_out])
    x = layer_norm(x + ff_np, p['gamma2'], p['beta2'])
    return x

def encoder(token_ids):
    x = embed(token_ids)
    for p in encoder_params:
        x = encoder_block(x, p)
    return x

tokens = np.random.randint(0, vocab_size, size=seq_len)
out = encoder(tokens)
print(out.shape)

(64, 64)


In [76]:
decoder_params = [{
    'W_Qs_m': [init_linear(d_model, d_k) for _ in range(h)],
    'W_Ks_m': [init_linear(d_model, d_k) for _ in range(h)],
    'W_Vs_m': [init_linear(d_model, d_k) for _ in range(h)],
    'W_O_m':  init_linear(d_model, d_model),
    'W_Qs_c': [init_linear(d_model, d_k) for _ in range(h)],
    'W_Ks_c': [init_linear(d_model, d_k) for _ in range(h)],
    'W_Vs_c': [init_linear(d_model, d_k) for _ in range(h)],
    'W_O_c':  init_linear(d_model, d_model),
    'W_f1': [[Value(np.random.randn()*0.01) for _ in range(d_model)] for _ in range(2*d_model)],
    'W_f2': [[Value(0.0) for _ in range(2*d_model)] for _ in range(d_model)],
    'b_f1': [Value(0.0) for _ in range(2*d_model)],
    'b_f2': [Value(0.0) for _ in range(d_model)],
    'gamma1': np.ones(d_model), 'beta1': np.zeros(d_model),
    'gamma2': np.ones(d_model), 'beta2': np.zeros(d_model),
    'gamma3': np.ones(d_model), 'beta3': np.zeros(d_model),
} for _ in range(N)]

mask = make_mask(seq_len)

def decoder_block(x, enc_out, p):
    x_list = [list(x[t]) for t in range(seq_len)]
    heads = []
    for i in range(h):
        Q = [linear(x_list[t], *p['W_Qs_m'][i]) for t in range(seq_len)]
        K = [linear(x_list[t], *p['W_Ks_m'][i]) for t in range(seq_len)]
        V = [linear(x_list[t], *p['W_Vs_m'][i]) for t in range(seq_len)]
        Q_np = np.array([[v.data for v in row] for row in Q])
        K_np = np.array([[v.data for v in row] for row in K])
        V_np = np.array([[v.data for v in row] for row in V])
        heads.append(masked_attention(Q_np, K_np, V_np, mask))
    concat  = [np.concatenate([heads[i][t] for i in range(h)]) for t in range(seq_len)]
    mha_out = [linear(list(concat[t]), *p['W_O_m']) for t in range(seq_len)]
    mha_np  = np.array([[v.data for v in row] for row in mha_out])
    x = layer_norm(x + mha_np, p['gamma1'], p['beta1'])
    x_list = [list(x[t]) for t in range(seq_len)]
    heads = []
    for i in range(h):
        Q = [linear(x_list[t], *p['W_Qs_c'][i]) for t in range(seq_len)]
        K = [linear(enc_out[t], *p['W_Ks_c'][i]) for t in range(seq_len)]
        V = [linear(enc_out[t], *p['W_Vs_c'][i]) for t in range(seq_len)]
        Q_np = np.array([[v.data for v in row] for row in Q])
        K_np = np.array([[v.data for v in row] for row in K])
        V_np = np.array([[v.data for v in row] for row in V])
        heads.append(attention(Q_np, K_np, V_np))
    concat   = [np.concatenate([heads[i][t] for i in range(h)]) for t in range(seq_len)]
    cross_out = [linear(list(concat[t]), *p['W_O_c']) for t in range(seq_len)]
    cross_np  = np.array([[v.data for v in row] for row in cross_out])
    x = layer_norm(x + cross_np, p['gamma2'], p['beta2'])
    x_list = [list(x[t]) for t in range(seq_len)]
    ff_out  = [feed_forward(x_list[t], p['W_f1'], p['b_f1'], p['W_f2'], p['b_f2']) for t in range(seq_len)]
    ff_np   = np.array([[v.data for v in row] for row in ff_out])
    x = layer_norm(x + ff_np, p['gamma3'], p['beta3'])
    return x

def decoder(token_ids, enc_out):
    x = embed(token_ids)
    for p in decoder_params:
        x = decoder_block(x, enc_out, p)
    return x

tgt_tokens = np.random.randint(0, vocab_size, size=seq_len)
out = decoder(tgt_tokens, out)
print(out.shape)

(64, 64)


In [77]:
W_final, b_final = init_linear(d_model, vocab_size)

def final_projection(x):
    logits = [linear(list(x[t]), W_final, b_final) for t in range(seq_len)]
    logits_np = np.array([[v.data for v in row] for row in logits])
    probs = softmax(logits_np)
    return logits_np, probs

logits, probs = final_projection(out)
print(logits.shape)
print(probs.shape)
print(probs[0].sum())

(64, 548)
(64, 548)
1.0
